In [8]:
!pip install -q transformers accelerate bitsandbytes sentence-transformers fastapi uvicorn nest-asyncio huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 24.1 MB/s eta 0:00:00


In [9]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [10]:
import os

BASE_DIR = "/content/drive/MyDrive/graph-rag/models"

LLM_MODEL_ID = "google/gemma-2b-it"
LLM_MODEL_DIR = os.path.join(BASE_DIR, "gemma-2b")

EMBEDDING_MODEL_ID = "google/embeddinggemma-300m"
EMBEDDING_MODEL_DIR = os.path.join(BASE_DIR, "embeddinggemma-300m")

os.makedirs(LLM_MODEL_DIR, exist_ok=True)
os.makedirs(EMBEDDING_MODEL_DIR, exist_ok=True)

print("LLM:", LLM_MODEL_DIR)
print("Embedding:", EMBEDDING_MODEL_DIR)

LLM: /content/drive/MyDrive/graph-rag/models/gemma-2b
Embedding: /content/drive/MyDrive/graph-rag/models/embeddinggemma-300m


In [ ]:
from huggingface_hub import login

login(token='huggingface token')

In [12]:
from huggingface_hub import snapshot_download

llm_config = os.path.join(LLM_MODEL_DIR, "config.json")

if os.path.exists(llm_config):
    print("Gemma 2B already exists in Google Drive.")
else:
    print("Gemma 2B not found. Downloading...")

    snapshot_download(
        repo_id=LLM_MODEL_ID,
        local_dir=LLM_MODEL_DIR
    )

    print("Gemma 2B downloaded.")

Gemma 2B already exists in Google Drive.


In [13]:
embedding_config = os.path.join(
    EMBEDDING_MODEL_DIR,
    "config.json"
)

if os.path.exists(embedding_config):
    print("EmbeddingGemma already exists in Google Drive.")
else:
    print("EmbeddingGemma not found. Downloading...")

    snapshot_download(
        repo_id=EMBEDDING_MODEL_ID,
        local_dir=EMBEDDING_MODEL_DIR
    )

    print("EmbeddingGemma downloaded.")

EmbeddingGemma already exists in Google Drive.


In [14]:
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16
)

llm_tokenizer = AutoTokenizer.from_pretrained(
    LLM_MODEL_DIR
)

llm_model = AutoModelForCausalLM.from_pretrained(
    LLM_MODEL_DIR,
    quantization_config=quantization_config,
    device_map="auto"
)

print("Gemma 2B loaded.")

Loading weights:   0%|          | 0/164 [00:00<?, ?it/s]

Gemma 2B loaded.


In [15]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    EMBEDDING_MODEL_DIR,
    device="cuda"
)

print("EmbeddingGemma loaded.")
print(
    "Embedding dimension:",
    embedding_model.get_sentence_embedding_dimension()
)

Loading weights:   0%|          | 0/314 [00:00<?, ?it/s]

EmbeddingGemma loaded.
Embedding dimension: 768


/tmp/ipykernel_1394/1610483108.py:11: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  embedding_model.get_sentence_embedding_dimension()


In [39]:
def generate_response(prompt):
    messages = [
        {"role": "user", "content": prompt}
    ]

    # Apply the chat template. This returns a dictionary with 'input_ids' and 'attention_mask'
    inputs = llm_tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt"
    )

    # Move the tensors from the dictionary to the GPU
    input_ids = inputs["input_ids"].to(llm_model.device)
    attention_mask = inputs["attention_mask"].to(llm_model.device)

    with torch.no_grad():
        outputs = llm_model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,   # Always good to pass this
            max_new_tokens=512,
            do_sample=True,
            temperature=0.7,
            pad_token_id=llm_tokenizer.eos_token_id
        )

    # Decode only the newly generated tokens (skip the prompt)
    generated_tokens = outputs[0][input_ids.shape[1]:]
    return llm_tokenizer.decode(generated_tokens, skip_special_tokens=True)
print(
    generate_response(
        "دانشگاه تهران در کدام شهر قرار دارد؟"
    )
)

دانشگاه تهران در شهر تهران، ایران قرار دارد. این دانشگاه در سال 1358 تأسیس شده و یکی از دانشگاه‌های قدیمی و مهمی ایران است.


In [40]:
def generate_embedding(text):
    embedding = embedding_model.encode(
        text,
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    return embedding.tolist()

embedding = generate_embedding(
    "دانشگاه تهران یکی از دانشگاه‌های ایران است."
)

print("Dimensions:", len(embedding))
print("First values:", embedding[:5])

Dimensions: 768
First values: [-0.03018031269311905, -0.023793213069438934, 0.019844016060233116, 0.023180274292826653, 0.052489109337329865]


In [41]:
from fastapi import FastAPI
from pydantic import BaseModel

app = FastAPI()


class GenerationRequest(BaseModel):
    prompt: str


class GenerationResponse(BaseModel):
    response: str


class EmbeddingRequest(BaseModel):
    text: str


class EmbeddingResponse(BaseModel):
    embedding: list[float]


@app.get("/health")
def health():
    return {
        "status": "ok"
    }


@app.post("/generate", response_model=GenerationResponse)
def generate(request: GenerationRequest):
    return {
        "response": generate_response(request.prompt)
    }


@app.post("/embed", response_model=EmbeddingResponse)
def embed(request: EmbeddingRequest):
    return {
        "embedding": generate_embedding(request.text)
    }

In [42]:
import nest_asyncio
import threading
import uvicorn

nest_asyncio.apply()

server_thread = threading.Thread(
    target=uvicorn.run,
    kwargs={
        "app": app,
        "host": "0.0.0.0",
        "port": 8000
    },
    daemon=True
)

server_thread.start()

print("FastAPI server started.")

FastAPI server started.


In [43]:
import requests

response = requests.get(
    "http://localhost:8000/health"
)

print(response.json())

INFO:     127.0.0.1:48444 - "GET /health HTTP/1.1" 200 OK
{'status': 'ok'}


In [44]:
response = requests.post(
    "http://localhost:8000/generate",
    json={
        "prompt": "دانشگاه تهران در کدام شهر قرار دارد؟"
    }
)

print(response.json())

INFO:     127.0.0.1:48454 - "POST /generate HTTP/1.1" 200 OK
{'response': 'دانشگاه تهران در شهر تهران قرار دارد. این دانشگاه در سال 1353 تأسیس شد و یکی از دانشگاه\u200cهای تاریخی و معتبر شده کشور است.'}


In [45]:
response = requests.post(
    "http://localhost:8000/embed",
    json={
        "text": "دانشگاه تهران یکی از دانشگاه‌های ایران است."
    }
)

result = response.json()

print("Embedding dimensions:", len(result["embedding"]))
print("First values:", result["embedding"][:5])

INFO:     127.0.0.1:52046 - "POST /embed HTTP/1.1" 200 OK
Embedding dimensions: 768
First values: [-0.03018031269311905, -0.023793213069438934, 0.019844016060233116, 0.023180274292826653, 0.052489109337329865]


In [23]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared

In [46]:
import subprocess
import time
import re
cloudflare_process = subprocess.Popen(
    [
        "./cloudflared",
        "tunnel",
        "--url",
        "http://localhost:8000"
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

time.sleep(5)

public_url = None

for _ in range(30):
    line = cloudflare_process.stdout.readline()

    if "trycloudflare.com" in line:
        match = re.search(
            r"https://[a-zA-Z0-9.-]+\.trycloudflare\.com",
            line
        )

        if match:
            public_url = match.group(0)
            break

print("Public API:", public_url)

Public API: https://licensed-wins-recording-whose.trycloudflare.com


In [47]:
import requests

response = requests.get(
    "http://localhost:8000/health",
    timeout=10
)

print(response.status_code)
print(response.json())

INFO:     127.0.0.1:40718 - "GET /health HTTP/1.1" 200 OK
200
{'status': 'ok'}


In [48]:
import requests

PUBLIC_URL = "https://licensed-wins-recording-whose.trycloudflare.com"

response = requests.get(
    f"{PUBLIC_URL}/health",
    timeout=30
)

print(response.status_code)
print(response.json())

INFO:     34.143.235.230:0 - "GET /health HTTP/1.1" 200 OK
200
{'status': 'ok'}
